# GBD_10 — Full 2-D basis sum (Era 2, step 2)

The 2-D analog of `GBD_6`, with the basis design carried forward from `GBD_8` (LiDAR-realistic parameters). Three things happen here:

1. Build a 9×9 = 81-element 2-D basis of on-axis Gaussian beamlets at width w = 0.4 mm, on a tensor-product grid of positions ranging ±2 mm in each axis.
2. Solve a regularized least-squares fit at z = 0 against a 2-D Gaussian source of waist w₀ = 1 mm.
3. Apply `GBD_9`'s per-beamlet propagator to every basis element, weight by the coefficients, sum, and visualize at multiple z.

**This is the first 2-D notebook that exercises the headline GBD claim: solve `c` once at z = 0, reuse at every other z.** GBD_9 verified one beamlet propagates correctly. GBD_10 verifies the sum propagates correctly.

**What is new vs. GBD_6 (1-D version):**
- Basis is a 2-D grid (81 elements) instead of a 1-D line (135 elements in GBD_6, 9 in GBD_8).
- G matrix is ~86 MB (vs. ~135 KB in GBD_6).
- Each propagator call operates on a 257×257 grid (vs. 1000 points in 1-D).
- Runtime is ~1 s per z value — fine for now, will revisit if GBD_11/12 demand vectorization.

**What is new vs. GBD_9 (single beamlet):** GBD_9 propagated one beamlet. Here we propagate 81 weighted beamlets and sum them. The propagator is unchanged; the new machinery is the basis-building loop, the regularized least-squares solve, and the weighted sum.

**Sanity checks performed below:**
1. z = 0 reconstruction L2 error vs. source Gaussian — should be in the 10⁻³ range (decomposition floor).
2. `propagate_basis(z=0)` must equal `G·c` to machine precision (propagator identity).
3. Propagated sum at z = zR should match the analytic 2-D Gaussian propagation of `w₀ = 1 mm` to the decomposition floor.
4. Multi-z snapshots — amplitude and phase at z = 0, zR/2, zR, 2zR, 5zR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# --- LiDAR-realistic parameters (carried forward from GBD_8/9) ---
wavelength = 1550e-9        # 1550 nm
w0_source  = 1.0e-3         # 1 mm — the source Gaussian waist
w_basis    = 0.4e-3         # 0.4 mm — each basis Gaussian's waist (smaller than source)
k          = 2*np.pi / wavelength
zR_source  = np.pi * w0_source**2 / wavelength    # ~2.027 m, source's Rayleigh range
zR_basis   = np.pi * w_basis**2 / wavelength      # ~0.324 m, basis Rayleigh range

# Paraxial validity (per CONCEPTS.md Topic 2 working agreement)
theta_div_source = wavelength / (np.pi * w0_source)
theta_div_basis  = wavelength / (np.pi * w_basis)
print(f'wavelength       = {wavelength*1e9:.1f} nm')
print(f'w0_source        = {w0_source*1e3:.3f} mm')
print(f'w_basis          = {w_basis*1e3:.3f} mm')
print(f'zR_source        = {zR_source:.4f} m')
print(f'zR_basis         = {zR_basis:.4f} m')
print(f'theta_div_source = {theta_div_source:.3e} rad   (paraxial OK)')
print(f'theta_div_basis  = {theta_div_basis:.3e} rad   (paraxial OK)')

# --- 2-D grid (sized like GBD_9: ±20 mm, N = 257 odd so center is exact origin) ---
Lx = Ly = 20e-3
Nx = Ny = 257
x = np.linspace(-Lx, Lx, Nx)
y = np.linspace(-Ly, Ly, Ny)
X, Y = np.meshgrid(x, y, indexing='xy')
print(f'\nGrid: {Nx} x {Ny} points, x in [{-Lx*1e3:.1f}, {Lx*1e3:.1f}] mm')

## The 2-D propagator (carried over unchanged from GBD_9)

Self-contained per project convention. Identical to GBD_9 — see CHANGELOG.md GBD_9 entry and CONCEPTS.md Topic 3 for the derivation.

In [ ]:
def propagate_tilted_gaussian_2d(X, Y, x0, y0, w0, kx, ky, k, z):
    """Propagate a 2-D tilted Gaussian beamlet by analytic q-parameter form."""
    wl = 2*np.pi / k
    zR_loc = np.pi * w0**2 / wl
    x_c = x0 + (kx/k) * z
    y_c = y0 + (ky/k) * z
    q0 = -1j * zR_loc
    qz = z + q0
    transverse = (q0/qz) * np.exp(1j * k * ((X-x_c)**2 + (Y-y_c)**2) / (2*qz))
    kz_long = np.sqrt(k**2 - kx**2 - ky**2)
    longitudinal = np.exp(1j * kz_long * z)
    tilt = np.exp(1j * (kx*X + ky*Y))
    return transverse * longitudinal * tilt

## Build the 2-D basis

Tensor-product grid: 9 positions in x times 9 positions in y, all on-axis (kx = ky = 0), all width `w_basis = 0.4 mm`. Spacing 0.5 mm in each axis, range ±2 mm. Mirrors GBD_8's 1-D choice but tensor-producted to 2-D.

`beam_info` is a list of (x0, y0, kx, ky) tuples — one per beamlet — that we'll iterate over both for the basis matrix at z = 0 and for the propagation step.

In [ ]:
n_per_axis = 9
spacing    = 0.5e-3   # 0.5 mm
x0_grid = (np.arange(n_per_axis) - (n_per_axis-1)/2) * spacing
y0_grid = (np.arange(n_per_axis) - (n_per_axis-1)/2) * spacing

beam_info = []
for x0i in x0_grid:
    for y0i in y0_grid:
        beam_info.append((x0i, y0i, 0.0, 0.0))   # on-axis: kx = ky = 0
N_beams = len(beam_info)
print(f'Basis size: {N_beams} = {n_per_axis}x{n_per_axis}')
print(f'x0 positions [mm]: {x0_grid*1e3}')
print(f'y0 positions [mm]: {y0_grid*1e3}')
print(f'Each beamlet: w = {w_basis*1e3:.3f} mm, kx = ky = 0')

## Build the G matrix at z = 0 and the source Gaussian

G has shape `(Nx*Ny, N_beams)` — one column per beamlet, each column a flattened 2-D field on the (x, y) grid.

Memory check: `257*257*81*16 bytes ≈ 86 MB`. Fits comfortably.

In [ ]:
t0 = time.time()

# Source: 2-D Gaussian centered at origin with waist w0_source
E_source = np.exp(-(X**2 + Y**2) / w0_source**2).astype(complex)

# Build G at z = 0 — one column per beamlet, flattened to 1-D
G = np.zeros((Nx*Ny, N_beams), dtype=complex)
for n, (x0n, y0n, kxn, kyn) in enumerate(beam_info):
    Eb = propagate_tilted_gaussian_2d(X, Y, x0n, y0n, w_basis, kxn, kyn, k, z=0.0)
    G[:, n] = Eb.ravel()

elapsed = time.time() - t0
print(f'Built G in {elapsed:.2f} s')
print(f'G shape: {G.shape}, dtype: {G.dtype}, memory: {G.nbytes/1e6:.1f} MB')

## Regularized least-squares solve

`(G†G + λ·I) c = G†·E_source`. Use `λ = 1e-8` (matching GBD_8 — basis is well-conditioned, no need for a large regularizer).

In [ ]:
lambda_reg = 1e-8

t0 = time.time()
GhG = G.conj().T @ G                       # 81 x 81
Ghe = G.conj().T @ E_source.ravel()        # 81
A   = GhG + lambda_reg * np.eye(N_beams)
c   = np.linalg.solve(A, Ghe)
elapsed = time.time() - t0
print(f'Solved {N_beams}x{N_beams} normal equations in {elapsed*1e3:.1f} ms')

# Reconstruction at z = 0
E_recon_z0 = (G @ c).reshape(Ny, Nx)

# Decomposition error (the floor — sets a lower bound on end-to-end accuracy at every z)
L2_recon = np.linalg.norm(E_recon_z0 - E_source) / np.linalg.norm(E_source)
print(f'Decomposition L2 relative error at z=0: {L2_recon:.3e}')
print(f'(GBD_8 1-D analog was 7.0e-3 — expect comparable here for 2-D)')

## propagate_basis function — the headline machinery

Given coefficients `c` solved at z = 0, propagate every beamlet by `z` and sum weighted by `c`. Mirrors GBD_6's 1-D version exactly, just with 2-D propagator and reshape at the end.

In [ ]:
def propagate_basis_2d(X, Y, beam_info, w_basis, k, z, coefficients):
    """Propagate every basis beamlet to distance z and weight-sum.
    Returns 2-D complex field of shape (Ny, Nx)."""
    Ny_loc, Nx_loc = X.shape
    Gp = np.zeros((Ny_loc*Nx_loc, len(beam_info)), dtype=complex)
    for n, (x0n, y0n, kxn, kyn) in enumerate(beam_info):
        Eb = propagate_tilted_gaussian_2d(X, Y, x0n, y0n, w_basis, kxn, kyn, k, z=z)
        Gp[:, n] = Eb.ravel()
    return (Gp @ coefficients).reshape(Ny_loc, Nx_loc)

## Sanity 2: propagator identity at z = 0

`propagate_basis_2d(z=0)` must reduce to `G·c`. Difference should be machine epsilon.

In [ ]:
E_prop_z0 = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=0.0, coefficients=c)
diff = np.max(np.abs(E_prop_z0 - E_recon_z0))
print(f'Max |propagate_basis(z=0) - G@c| = {diff:.3e}')
print('(should be ~1e-15)')

## Sanity 3: propagated sum at z = zR vs. analytic 2-D Gaussian

The source is a clean 2-D Gaussian with `w₀ = 1 mm`. Its analytic propagation to z is a 2-D Gaussian with `w(z) = w₀·sqrt(1+(z/zR_source)²)`. We compare the GBD-propagated sum against this analytic answer at z = zR_source.

**Important conceptual point:** the basis beamlets each have `zR_basis ≈ 0.324 m`, but the source has `zR_source ≈ 2.027 m`. We're propagating to z = zR_source, well beyond the basis Rayleigh range — and that's fine, because the *envelope* of the basis sum is the source, not any individual beamlet. The basis is scaffolding.

In [ ]:
z_test = zR_source

# Analytic propagation of the source Gaussian
E_analytic = propagate_tilted_gaussian_2d(X, Y, 0, 0, w0_source, 0, 0, k, z_test)

# GBD propagation of the basis sum
t0 = time.time()
E_gbd = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=z_test, coefficients=c)
elapsed = time.time() - t0
print(f'propagate_basis_2d to z=zR took {elapsed:.2f} s')

# Compare
L2_prop = np.linalg.norm(E_gbd - E_analytic) / np.linalg.norm(E_analytic)
print(f'L2 rel |E_gbd - E_analytic| at z=zR: {L2_prop:.3e}')
print(f'(should be comparable to decomposition floor {L2_recon:.3e})')

## Multi-z snapshots — amplitude and phase

Top row: |E_gbd|. Bottom row: arg(E_gbd) with bulk longitudinal phase factored out. The propagated GBD sum at each z.

In [ ]:
z_values = [0.0, zR_source/2, zR_source, 2*zR_source, 5*zR_source]
fields = []
t0 = time.time()
for z_v in z_values:
    E_v = propagate_basis_2d(X, Y, beam_info, w_basis, k, z=z_v, coefficients=c)
    # subtract bulk longitudinal phase to keep arg(E) in (-pi, pi)
    kz_loc = k   # on-axis sum — kz reduces to k for kx = ky = 0
    E_env = E_v * np.exp(-1j*kz_loc*z_v)
    fields.append((z_v, E_v, E_env))
elapsed = time.time() - t0
print(f'Propagated to {len(z_values)} z values in {elapsed:.2f} s')

fig, axes = plt.subplots(2, len(z_values), figsize=(4*len(z_values), 7))
extent = [-Lx*1e3, Lx*1e3, -Ly*1e3, Ly*1e3]
for i, (z_v, E_v, E_env) in enumerate(fields):
    ax = axes[0, i]
    im = ax.imshow(np.abs(E_env), origin='lower', extent=extent, cmap='viridis')
    w_z = w0_source * np.sqrt(1 + (z_v/zR_source)**2) if z_v > 0 else w0_source
    ax.set_title(f'|E_gbd|, z = {z_v:.3f} m\nw_source(z) = {w_z*1e3:.3f} mm')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax = axes[1, i]
    im = ax.imshow(np.angle(E_env), origin='lower', extent=extent, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'arg(E_gbd), z = {z_v:.3f} m')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Summary

If all sanity checks pass:
- Sanity 1: decomposition L2 rel ~ 10⁻³ at z = 0 (decomposition floor).
- Sanity 2: propagator identity at z = 0 to ~1e-15.
- Sanity 3: propagated GBD sum at z = zR matches analytic 2-D Gaussian to within decomposition floor.
- Multi-z plot: amplitude shows clean Gaussian envelope at every z, broadening per `w(z)`. Phase shows the expected curvature.

Then the 2-D basis-sum machinery is verified, and `GBD_11` can compare against an independent 2-D angular-spectrum FFT and sweep z over the LiDAR operating range.

**What this notebook deliberately did not do** (saving for later):
- 2-D angular-spectrum FFT reference — `GBD_11`.
- Sweep z to 100 m — `GBD_11`.
- Non-Gaussian or off-axis sources — `GBD_12` (conditional).
- Vectorized propagator — premature optimization.